# Notebook 1 — Step 1 Setup environment & Step 2 Load dataset
| | |
|---|---|
| **Case Study** | 6 — Smart City IoT Data Platform (6.1, 6.3 Real-time pipelines, ML systems) |
| **Roll No** | `<ROLL_NO>` |
| **Name** | `<NAME>` |

**Goal:** start Spark, then load the three IoT sources in three formats:

| Source | Format | Rows | What it simulates |
|---|---|---|---|
| Traffic counters (Kaggle *Smart City Traffic Patterns*) | **CSV** | 48,120 | 4 junction sensors, hourly vehicle counts, Nov 2015 – Jun 2017 |
| Air-quality monitor (UCI *Air Quality*) | **CSV** (`;` separator, `,` decimals) | 9,357 | multi-gas sensor station, hourly, Mar 2004 – Apr 2005 |
| Weather station (Open-Meteo) | **API → JSON** | 192 / call | live weather for the city |


## Step 1 — Environment

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pyspark.sql import functions as F
from src import config
from src.spark_utils import get_spark
spark = get_spark("NB01-SetupLoad")
print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

In [ ]:
import platform, subprocess
java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
print("Python      :", platform.python_version())
print("Java        :", java)
print("PySpark     :", spark.version)
print("Master      :", spark.sparkContext.master, "| cores:", spark.sparkContext.defaultParallelism)
print("HADOOP_HOME :", os.environ.get("HADOOP_HOME", "(not needed on Linux/macOS)"))
for k in ["spark.driver.memory", "spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled", "spark.sql.session.timeZone"]:
    print(f"{k:32s}", spark.conf.get(k))

> 📸 **Screenshot 1:** the output above, plus the Spark UI (the URL printed at the top) → *Environment* tab.

## Step 2 — Load dataset
### 2a. Traffic CSV with an explicit schema
`inferSchema=True` makes Spark read the whole file once just to guess the types. An explicit schema skips that pass, and a value with the wrong type becomes a `null` that we can count, instead of a silent type change.

In [ ]:
import time
from src import ingest

t0 = time.perf_counter()
inferred = spark.read.option("header", True).option("inferSchema", True).csv(config.p(config.TRAFFIC_TRAIN_CSV))
inferred.count()
t_infer = time.perf_counter() - t0

t0 = time.perf_counter()
traffic_raw = ingest.load_traffic_raw(spark)
traffic_raw.count()
t_schema = time.perf_counter() - t0
print(f"inferSchema: {t_infer:.2f}s   explicit schema: {t_schema:.2f}s")
traffic_raw.printSchema()
traffic_raw.show(5)

In [ ]:
traffic = ingest.clean_traffic(traffic_raw)
print("rows:", traffic.count())
traffic.groupBy("junction").agg(F.min("event_time").alias("first"), F.max("event_time").alias("last"),
                                F.count("*").alias("rows"), F.round(F.avg("vehicles"), 1).alias("avg_vehicles")).orderBy("junction").show()

### 2b. Air-quality CSV: a messy real-world file
Problems in this file and how we handle them:
* separator is `;` → `sep=";"`
* decimals use a comma (`2,6`) → read as text, replace `,` with `.`, then cast to double
* missing values are coded as **-200** → turned into `null`
* 114 blank lines at the end, and two empty columns (`;;`) → dropped
* `NMHC(GT)` is ~90% missing → column dropped

In [ ]:
air_raw = ingest.load_air_raw(spark)
air_raw.select("Date", "Time", "`CO(GT)`", "`PT08.S1(CO)`", "T", "RH").show(3)
air = ingest.clean_air(air_raw)
air.printSchema()
air.select("event_time", "co_gt", "pt08_s1_co", "nox_gt", "temperature", "rel_humidity").show(5)

In [ ]:
missing = air.select([F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 1).alias(c) for c in air.columns])
print("% missing per column after cleaning:")
missing.show(vertical=True)

### 2c. Weather API → JSON
Open-Meteo is free and needs no API key. The raw response is saved to `data/raw/weather/` (so the run can be reproduced) and then loaded with `spark.read.json`. The JSON is *column-oriented* (one array per variable), so `arrays_zip` + `explode` turn it into one row per hour.
If the API is unreachable, the code falls back to the last saved response.

In [ ]:
path, source = ingest.latest_weather_json()
print("weather data from:", source, "->", path.name)
raw_json = spark.read.option("multiLine", True).json(config.p(path))
raw_json.printSchema()
weather = ingest.load_weather(spark, path)
weather.show(5)

### 2d. Persist all layers (bronze → silver → gold) and the data-quality report
`src/batch_pipeline.py` runs the same steps as above and writes them as Parquet:
* **bronze** = raw data as loaded
* **silver** = cleaned and typed
* **gold** = model-ready features and aggregates

In [ ]:
from src import batch_pipeline
report = batch_pipeline.run(spark, verbose=False)
import json; print(json.dumps(report, indent=1, default=str)[:2500])

> 📸 **Screenshot 2:** the schemas, `show()` outputs and the data-quality report above.

In [ ]:
spark.stop()